In [1]:
# 1. Imports
import sys
sys.path.append("..")

import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from src.preprocessing import preprocess

# 2. Load raw data
raw_train = pd.read_csv("../data/raw/train.csv")
raw_test = pd.read_csv("../data/raw/test.csv")

# 3. Compute the three "learned from train only" lookup tables, BEFORE any preprocessing call
temp_title = raw_train["Name"].str.extract(r",\s*([^.]+)\.")[0]
temp_title = temp_title.where(temp_title.isin(["Mr", "Miss", "Mrs", "Master"]), "Rare")
age_medians_by_title = raw_train["Age"].groupby(temp_title).median()

fare_medians_by_pclass = raw_train.groupby("Pclass")["Fare"].median()

combined_ticket_counts = pd.concat([raw_train["Ticket"], raw_test["Ticket"]]).value_counts()

# 4. NOW preprocess both train and test, using those same fixed lookup tables
train_processed = preprocess(raw_train, age_medians_by_title, combined_ticket_counts, fare_medians_by_pclass)
test_processed = preprocess(raw_test, age_medians_by_title, combined_ticket_counts, fare_medians_by_pclass)

# 5. Build X, y from the processed train data
feature_cols = [
    "IsFemale", "Pclass", "Age", "Fare", "FamilySize", "HasCabin",
    "Title_Master", "Title_Miss", "Title_Mr", "Title_Mrs", "Title_Rare",
    "Embarked_C", "Embarked_Q", "Embarked_S", "TicketGroupSize"
]
X = train_processed[feature_cols]
y = train_processed["Survived"]
X_test_final = test_processed[feature_cols]

# 6. Sanity check before training anything
print(X.isna().sum().sum(), "missing values in X")
print(X_test_final.isna().sum().sum(), "missing values in X_test_final")

0 missing values in X
0 missing values in X_test_final


In [2]:
final_model = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42)
final_model.fit(X, y)

test_predictions = final_model.predict(X_test_final)

In [3]:
submission = pd.DataFrame({
    "PassengerId": raw_test["PassengerId"],
    "Survived": test_predictions.astype(int)
})
submission.to_csv("../data/processed/submission.csv", index=False)

In [4]:
submission.head()

,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,1


In [5]:
submission.shape
submission["Survived"].value_counts()

Survived
0    271
1    147
Name: count, dtype: int64

In [7]:
test_processed[["Pclass", "IsFemale", "Title_Mr"]].describe(include="all")

,Pclass,IsFemale,Title_Mr
count,418.000000,418,418
unique,NaN,2,2
top,NaN,False,True
freq,NaN,266,240
mean,2.265550,NaN,NaN
std,0.841838,NaN,NaN
min,1.000000,NaN,NaN
25%,1.000000,NaN,NaN
50%,3.000000,NaN,NaN
75%,3.000000,NaN,NaN


In [9]:
from sklearn.linear_model import LogisticRegression

final_lr_model = LogisticRegression(max_iter=1000)
final_lr_model.fit(X, y)

lr_predictions = final_lr_model.predict(X_test_final)

lr_submission = pd.DataFrame({
    "PassengerId": raw_test["PassengerId"],
    "Survived": lr_predictions.astype(int)
})
lr_submission.to_csv("../data/processed/submission_logreg.csv", index=False)